In [24]:
%pip install -U "transformers>=4.41" "datasets>=2.20" "accelerate>=0.33" "evaluate>=0.4" sentencepiece sacrebleu tqdm matplotlib
%pip install sacremoses
%pip install -U ftfy unidecode langid "clean-text[gpl]"
%pip install -U transformers


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import torch
print(torch.cuda.memory_summary(device=None, abbreviated=True))

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   4726 MiB |  12773 MiB |  14563 GiB |  14558 GiB |
|---------------------------------------------------------------------------|
| Active memory         |   4726 MiB |  12773 MiB |  14563 GiB |  14558 GiB |
|---------------------------------------------------------------------------|
| Requested memory      |   4726 MiB |  12768 MiB |  14477 GiB |  14472 GiB |
|---------------------------------------------------------------

In [ ]:
import os, random, numpy as np, torch, inspect
import matplotlib.pyplot as plt
from datasets import load_dataset
import evaluate
from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer,
)

def forced_bos_id(tok, lang_code: str):
    m = getattr(tok, "lang_code_to_id", None)
    if isinstance(m, dict) and lang_code in m:
        return m[lang_code]
    if hasattr(tok, "get_lang_id"):
        return tok.get_lang_id(lang_code)
    i = tok.convert_tokens_to_ids(lang_code)
    if i is None or i == getattr(tok, "unk_token_id", None):
        raise RuntimeError(f"Could not resolve id for {lang_code}. Try upgrading `transformers`.")
    return i

# Repro
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    try:
        torch.set_float32_matmul_precision("high")
        torch.backends.cuda.matmul.allow_tf32 = True
    except Exception:
        pass

device = "cuda" if torch.cuda.is_available() else "cpu"
use_bf16 = torch.cuda.is_available() and getattr(torch.cuda, "is_bf16_supported", lambda: False)()
use_fp16 = torch.cuda.is_available() and not use_bf16
print(f"Device={device} | bf16={use_bf16} | fp16={use_fp16}")

# Project + knobs
PROJECT_DIR = "./nllb_bilingual_en_es_pt"
os.makedirs(PROJECT_DIR, exist_ok=True)

ROOT = Path(r"J:\FINAL PROJECT")

TMX_ES = ROOT / "data" / "en-es.tmx"   

TMX_PT = ROOT / "data" / "en-pt.tmx"    

print("TMX_ES =", TMX_ES)
print("TMX_PT =", TMX_PT)
print("exists:", TMX_ES.exists(), TMX_PT.exists())

MAX_SAMPLES = 1200        # per split per direction (train subset)
EVAL_SAMPLES = 300        # validation subset
MAX_SRC_LEN = 64
MAX_TGT_LEN = 64
BATCH = 8
GRAD_ACCUM = 1
MAX_STEPS = 500           # bump to 1500–3000 for better quality




Device=cuda | bf16=True | fp16=False
TMX_ES = J:\FINAL PROJECT\data\en-es.tmx
TMX_PT = J:\FINAL PROJECT\data\en-pt.tmx
exists: True True


c:\Users\jeeva\anaconda3\Lib\site-packages\torch\__init__.py:1628: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:50.)
  _C._set_float32_matmul_precision(precision)


In [ ]:
# --- TMX loader ---
import io, gzip, unicodedata, regex as re
from xml.etree import ElementTree as ET
from datasets import Dataset

def _nfc_ws(s: str) -> str:
    s = unicodedata.normalize("NFC", s or "")
    s = s.replace("\u00A0"," ").replace("\u202F"," ")
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def _norm_lang(tag: str) -> str:
    if not tag: return ""
    t = tag.replace("_","-").lower()
    # normalize to 2-letter where possible (tmx can have eng/spa/por)
    if t.startswith("eng"): return "en"
    if t.startswith("spa"): return "es"
    if t.startswith("por"): return "pt"
    return t.split("-")[0]

def _open_maybe_gz(path: str):
    return io.TextIOWrapper(gzip.open(path, "rb"), encoding="utf-8", errors="ignore") if path.endswith(".gz") \
           else io.open(path, "r", encoding="utf-8", errors="ignore")

def load_tmx_as_dataset(tmx_path: str, src="en", tgt="es", max_samples: int=None, seed: int=42) -> Dataset:
    p = Path(tmx_path); assert p.exists(), f"Missing TMX: {tmx_path}"
    src = _norm_lang(src); tgt = _norm_lang(tgt)
    pairs = []
    with _open_maybe_gz(str(p)) as fh:
        context = ET.iterparse(fh, events=("start","end"))
        _, root = next(context)
        cur_src, cur_tgt = None, None
        for event, elem in context:
            tag = elem.tag.lower()
            if event == "start" and tag.endswith("tu"):
                cur_src, cur_tgt = None, None
            if event == "end" and tag.endswith("tuv"):
                lang = _norm_lang(elem.attrib.get("{http://www.w3.org/XML/1998/namespace}lang", elem.attrib.get("lang","")))
                seg = next((c for c in elem if c.tag.lower().endswith("seg")), None)
                if seg is not None:
                    text = _nfc_ws("".join(seg.itertext()))
                    if lang == src and text: cur_src = text
                    elif lang == tgt and text: cur_tgt = text
            if event == "end" and tag.endswith("tu"):
                if cur_src and cur_tgt:
                    pairs.append((cur_src, cur_tgt))
                root.clear()
                if max_samples and len(pairs) >= max_samples:
                    break
    if max_samples and len(pairs) > max_samples:
        import random
        rnd = random.Random(seed)
        idx = list(range(len(pairs))); rnd.shuffle(idx); idx = idx[:max_samples]
        pairs = [pairs[i] for i in idx]
    return Dataset.from_dict({"src_text":[s for s,_ in pairs], "tgt_text":[t for _,t in pairs]})
# --- end TMX loader ---


In [ ]:
print("⏳ Loading TMX…")
# Load EN→ES from your TMX
ds_es_all = load_tmx_as_dataset(str(TMX_ES), src="en", tgt="es",
                                max_samples=MAX_SAMPLES+EVAL_SAMPLES, seed=SEED).shuffle(seed=SEED)
split_es = ds_es_all.train_test_split(test_size=EVAL_SAMPLES, seed=SEED)
train_es, valid_es = split_es["train"], split_es["test"]

# Load EN→PT from your TMX
ds_pt_all = load_tmx_as_dataset(str(TMX_PT), src="en", tgt="pt",
                                max_samples=MAX_SAMPLES+EVAL_SAMPLES, seed=SEED).shuffle(seed=SEED)
split_pt = ds_pt_all.train_test_split(test_size=EVAL_SAMPLES, seed=SEED)
train_pt, valid_pt = split_pt["train"], split_pt["test"]

print("Train sizes  EN→ES / EN→PT:", len(train_es), len(train_pt))
print("Valid sizes  EN→ES / EN→PT:", len(valid_es), len(valid_pt))

⏳ Loading TMX…
Train sizes  EN→ES / EN→PT: 1200 1200
Valid sizes  EN→ES / EN→PT: 300 300


In [ ]:
CLEAN = False 

if CLEAN:
    import regex as re, unicodedata, numpy as np
    import langid
    langid.set_languages(['en','es','pt'])

    URL_RE = re.compile(r"(https?://\S+|www\.\S+)")
    PUNCT_ONLY_RE = re.compile(r"^\p{P}+$", re.UNICODE)

    def nfc(s): 
        s = unicodedata.normalize("NFC", s or "")
        return s.replace("\u00A0"," ").replace("\u202F"," ").strip()

    def collapse_ws(s): return re.sub(r"\s+", " ", s).strip()

    def too_short_or_punct(s): 
        return (not s) or PUNCT_ONLY_RE.match(s) or len(s.split()) < 2

    def looks_like_en(text, thr=0.85):   # 🔸 looser & source-only
        if not text: return False
        lang, conf = langid.classify(text)
        return (lang == "en") and (conf >= thr)

    def ratio_ok(src, tgt, lo=0.25, hi=4.0):  # 🔸 wider than before
        sl, tl = max(1,len(src.split())), max(1,len(tgt.split()))
        r = sl/tl
        return lo <= r <= hi

    def clean_example(ex):
        src = URL_RE.sub("<URL>", collapse_ws(nfc(ex["src_text"])))
        tgt = URL_RE.sub("<URL>", collapse_ws(nfc(ex["tgt_text"])))
        return {"src_text": src, "tgt_text": tgt}

    def basic_quality_ok(ex):
        src, tgt = ex["src_text"], ex["tgt_text"]
        if too_short_or_punct(src) or too_short_or_punct(tgt): return False
        if len(src.split()) > 200 or len(tgt.split()) > 200: return False   # 🔸 allow longer
        if not ratio_ok(src, tgt): return False
        if not looks_like_en(src): return False                               # 🔸 only check EN
        return True

    def dedup_pairs(ds):
        seen, keep = set(), []
        for i, ex in enumerate(ds):
            key = (collapse_ws(ex["src_text"]).lower(), collapse_ws(ex["tgt_text"]).lower())
            if key not in seen:
                seen.add(key); keep.append(i)
        return ds.select(keep)

    # ----- Clean + dedup the WHOLE corpus BEFORE split -----
    print("[ES] pre-clean size:", len(ds_es_all))
    ds_es_all = ds_es_all.map(clean_example)
    ds_es_all = dedup_pairs(ds_es_all)
    print("[ES] post-dedup size:", len(ds_es_all))

    print("[PT] pre-clean size:", len(ds_pt_all))
    ds_pt_all = ds_pt_all.map(clean_example)
    ds_pt_all = dedup_pairs(ds_pt_all)
    print("[PT] post-dedup size:", len(ds_pt_all))

    # ----- Split (keep eval small & proportional if corpora are small) -----
    es_eval = min(EVAL_SAMPLES, max(50, int(0.1 * len(ds_es_all))))  # 10% or >=50
    pt_eval = min(EVAL_SAMPLES, max(50, int(0.1 * len(ds_pt_all))))

    split_es = ds_es_all.train_test_split(test_size=es_eval, seed=SEED, shuffle=True)
    split_pt = ds_pt_all.train_test_split(test_size=pt_eval, seed=SEED, shuffle=True)

    train_es, valid_es = split_es["train"], split_es["test"]
    train_pt, valid_pt = split_pt["train"], split_pt["test"]

    print("[ES] split sizes (raw):", len(train_es), len(valid_es))
    print("[PT] split sizes (raw):", len(train_pt), len(valid_pt))

    # ----- Gentle quality filters AFTER split (no overlap removal) -----
    train_es = train_es.filter(basic_quality_ok)
    valid_es = valid_es.filter(basic_quality_ok)
    train_pt = train_pt.filter(basic_quality_ok)
    valid_pt = valid_pt.filter(basic_quality_ok)

    print("[ES] after filters:", len(train_es), len(valid_es))
    print("[PT] after filters:", len(train_pt), len(valid_pt))

    # ----- Safety net: if a split got empty, fall back to unfiltered -----
    if len(train_es) == 0:
        print("⚠️ ES train became empty; using pre-filter train split.")
        train_es = split_es["train"]
    if len(valid_es) == 0:
        print("⚠️ ES valid became empty; using pre-filter valid split.")
        valid_es = split_es["test"]

    if len(train_pt) == 0:
        print("⚠️ PT train became empty; using pre-filter train split.")
        train_pt = split_pt["train"]
    if len(valid_pt) == 0:
        print("⚠️ PT valid became empty; using pre-filter valid split.")
        valid_pt = split_pt["test"]

    print("✅ Final sizes EN→ES / EN→PT:",
          len(train_es), len(valid_es), "|", len(train_pt), len(valid_pt))


In [ ]:
from transformers import AutoTokenizer

nllb_ckpt = "facebook/nllb-200-distilled-600M"
SRC_LANG = "eng_Latn"
ES_LANG  = "spa_Latn"
PT_LANG  = "por_Latn"

GLOSSARY = {
    "polymerase chain reaction": {"es": "reacción en cadena de la polimerasa", "pt": "reação em cadeia da polimerase"},
    "confidence interval": {"es": "intervalo de confianza", "pt": "intervalo de confiança"},
}

def apply_glossary(text: str, lang_short: str) -> str:
    out = text
    for en_term, m in GLOSSARY.items():
        if lang_short in m:
            out = out.replace(en_term, m[lang_short])
    return out

def preprocess_any(batch, tok, tgt_short: str):
    targets = [apply_glossary(t, tgt_short) for t in batch["tgt_text"]]
    model_inputs = tok(batch["src_text"], truncation=True, max_length=MAX_SRC_LEN)
    try:
        labels = tok(text_target=targets, truncation=True, max_length=MAX_TGT_LEN)
    except TypeError:
        from contextlib import contextmanager
        cm = tok.as_target_tokenizer() if hasattr(tok, "as_target_tokenizer") else contextmanager(lambda: (yield))()
        with cm:
            labels = tok(targets, truncation=True, max_length=MAX_TGT_LEN)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Tokenizers (created just for tokenization phase)
tok_es_tmp = AutoTokenizer.from_pretrained(nllb_ckpt, use_fast=True, src_lang=SRC_LANG, tgt_lang=ES_LANG)
tok_pt_tmp = AutoTokenizer.from_pretrained(nllb_ckpt, use_fast=True, src_lang=SRC_LANG, tgt_lang=PT_LANG)

train_es_tok = train_es.map(lambda b: preprocess_any(b, tok_es_tmp, "es"),
                            batched=True, remove_columns=train_es.column_names)
valid_es_tok = valid_es.map(lambda b: preprocess_any(b, tok_es_tmp, "es"),
                            batched=True, remove_columns=valid_es.column_names)

train_pt_tok = train_pt.map(lambda b: preprocess_any(b, tok_pt_tmp, "pt"),
                            batched=True, remove_columns=train_pt.column_names)
valid_pt_tok = valid_pt.map(lambda b: preprocess_any(b, tok_pt_tmp, "pt"),
                            batched=True, remove_columns=valid_pt.column_names)



Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [ ]:
from pathlib import Path
import json

CACHE_DIR = Path("cache_tok")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

meta = {
    "tokenizer": "facebook/nllb-200-distilled-600M",
    "src_lang": SRC_LANG,
    "es_lang": ES_LANG,
    "pt_lang": PT_LANG,
    "max_src_len": MAX_SRC_LEN,
    "max_tgt_len": MAX_TGT_LEN,
}
with open(CACHE_DIR / "_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

def safe_save(ds, path):
    if len(ds) == 0:
        print(f"⚠️ Skipping save for {path}: dataset is empty")
        return
    ds.save_to_disk(str(path))

safe_save(train_es_tok, CACHE_DIR / "train_es")
safe_save(valid_es_tok, CACHE_DIR / "valid_es")
safe_save(train_pt_tok, CACHE_DIR / "train_pt")
safe_save(valid_pt_tok, CACHE_DIR / "valid_pt")

print("✅ Tokenized datasets cached in", CACHE_DIR)


Saving the dataset (0/1 shards):   0%|          | 0/1200 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/300 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1200 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/300 [00:00<?, ? examples/s]

✅ Tokenized datasets cached in cache_tok


In [ ]:
from transformers import TrainingArguments

def make_training_args(**wanted):
    accepted = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
    filtered = {k: v for k, v in wanted.items() if k in accepted}
    dropped = sorted(set(wanted) - set(filtered))
    if dropped: print("Dropped unsupported keys:", dropped)
    return TrainingArguments(**filtered)

args_es = make_training_args(
    output_dir=f"{PROJECT_DIR}/ckpt_es",
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    max_steps=MAX_STEPS,
    learning_rate=3e-5,
    lr_scheduler_type="linear",
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=100,
    report_to="none",
    bf16=use_bf16, fp16=use_fp16,
    no_cuda=(device != "cuda"),
    dataloader_num_workers=0,
    dataloader_pin_memory=(device == "cuda"),
    eval_strategy="no",
    save_strategy="no",
    optim="adamw_torch_fused",
    group_by_length=True,
)

args_pt = make_training_args(
    output_dir=f"{PROJECT_DIR}/ckpt_pt",
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    max_steps=MAX_STEPS,
    learning_rate=3e-5,
    lr_scheduler_type="linear",
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=100,
    report_to="none",
    bf16=use_bf16, fp16=use_fp16,
    no_cuda=(device != "cuda"),
    dataloader_num_workers=0,
    dataloader_pin_memory=(device == "cuda"),
    eval_strategy="no",
    save_strategy="no",
    optim="adamw_torch_fused",
    group_by_length=True,
)


In [ ]:
# after you create tokenizer + model
tok_pt.src_lang = SRC_LANG
tok_pt.tgt_lang = PT_LANG
lang_id_pt = forced_bos_id(tok_pt, PT_LANG)
mod_pt.config.forced_bos_token_id = lang_id_pt
mod_pt.config.decoder_start_token_id = lang_id_pt
mod_pt.generation_config.forced_bos_token_id = lang_id_pt
mod_pt.generation_config.decoder_start_token_id = lang_id_pt


In [ ]:
print("🔎 Evaluating EN→PT…")
_ = evaluate_pair(mod_pt, tok_pt, valid_pt, name="EN→PT")


🔎 Evaluating EN→PT…
EN→PT -> BLEU 0.01 | chrF 0.89 | exact 0.00%  on 300 ex.


In [28]:

bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

EVAL_LIMIT   = 100 
MAX_NEW_TOK  = 64
MIN_NEW_TOK  = 3
NUM_BEAMS    = 4
BATCH_SIZE   = 32

@torch.inference_mode()
def batch_generate(model, tok, texts, max_len_src=128, max_new=MAX_NEW_TOK, beams=NUM_BEAMS, batch_size=BATCH_SIZE):
    enc = tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=max_len_src)
    input_ids = enc["input_ids"].to(model.device)
    attn = enc.get("attention_mask")
    if attn is not None:
        attn = attn.to(model.device)

    preds = []
    for i in range(0, input_ids.size(0), batch_size):
        sl = slice(i, i + batch_size)
        out = model.generate(
            input_ids=input_ids[sl],
            attention_mask=(attn[sl] if attn is not None else None),
            max_new_tokens=max_new,
            min_new_tokens=MIN_NEW_TOK,
            num_beams=beams,
            no_repeat_ngram_size=3,
            length_penalty=1.0,
            do_sample=False,
            forced_bos_token_id=model.config.forced_bos_token_id,
            decoder_start_token_id=model.config.decoder_start_token_id,  # << added
        )
        preds.extend(tok.batch_decode(out, skip_special_tokens=True))
    return [p.strip() for p in preds]


def evaluate_pair(model, tok, valid_ds, name="EN→X"):
    src = [r["src_text"] for r in valid_ds][:EVAL_LIMIT]
    refs = [r["tgt_text"] for r in valid_ds][:EVAL_LIMIT]
    preds = batch_generate(model, tok, src, max_len_src=MAX_SRC_LEN)
    refs_bleu = [[r] for r in refs]
    bleu = bleu_metric.compute(predictions=preds, references=refs_bleu)["score"]
    chrf = chrf_metric.compute(predictions=preds, references=refs_bleu)["score"]
    exact = float(np.mean([int(p == r) for p, r in zip(preds, refs)]) * 100.0)
    print(f"{name} -> BLEU {bleu:.2f} | chrF {chrf:.2f} | exact {exact:.2f}%  on {len(preds)} ex.")
    return {"bleu": bleu, "chrf": chrf, "exact_acc": exact}


In [29]:
print("🔹 Loading EN→ES model...")
tok_es = AutoTokenizer.from_pretrained(nllb_ckpt, use_fast=True, src_lang=SRC_LANG, tgt_lang=ES_LANG)
mod_es = AutoModelForSeq2SeqLM.from_pretrained(nllb_ckpt)

# try GPU; fallback CPU if OOM
try:
    tok_es.src_lang = SRC_LANG
    tok_es.tgt_lang = ES_LANG
    _es_id = forced_bos_id(tok_es, ES_LANG)
    mod_es.config.forced_bos_token_id = _es_id
    mod_es.config.decoder_start_token_id = _es_id
    mod_es.generation_config.forced_bos_token_id = _es_id
    mod_es.generation_config.decoder_start_token_id = _es_id

    
except RuntimeError as e:
    if "CUDA out of memory" in str(e):
        print("⚠️ CUDA OOM moving EN→ES to GPU. Falling back to CPU.")
        mod_es = mod_es.to("cpu")
    else:
        raise

mod_es.config.forced_bos_token_id = forced_bos_id(tok_es, ES_LANG)
try: mod_es.gradient_checkpointing_enable()
except Exception: pass

collator_es = DataCollatorForSeq2Seq(tokenizer=tok_es, model=mod_es, padding="longest")
trainer_es = Trainer(
    model=mod_es,
    args=args_es,
    train_dataset=train_es_tok,
    eval_dataset=None,
    data_collator=collator_es,
    tokenizer=tok_es,
)

print("✅ EN→ES trainer ready. Starting training…")
train_output_es = trainer_es.train()
print("✅ EN→ES training done.")

print("🔎 Evaluating EN→ES…")
_ = evaluate_pair(mod_es, tok_es, valid_es, name="EN→ES")

# Free VRAM
import gc
del trainer_es, collator_es, train_output_es
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache(); torch.cuda.synchronize()


🔹 Loading EN→ES model...


C:\Users\jeeva\AppData\Local\Temp\ipykernel_36288\1684949385.py:28: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_es = Trainer(


✅ EN→ES trainer ready. Starting training…


Step,Training Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 1002.00 MiB. GPU 0 has a total capacity of 7.96 GiB of which 0 bytes is free. Of the allocated memory 20.73 GiB is allocated by PyTorch, and 1.09 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
print("🔹 Loading EN→PT model...")
tok_pt = AutoTokenizer.from_pretrained(nllb_ckpt, use_fast=True, src_lang=SRC_LANG, tgt_lang=PT_LANG)
mod_pt = AutoModelForSeq2SeqLM.from_pretrained(nllb_ckpt)

try:
    # Force Portuguese decoding always
    tok_pt.src_lang = SRC_LANG
    tok_pt.tgt_lang = PT_LANG
    _pt_id = forced_bos_id(tok_pt, PT_LANG)
    mod_pt.config.forced_bos_token_id = _pt_id
    mod_pt.config.decoder_start_token_id = _pt_id
    od_pt.generation_config.forced_bos_token_id = _pt_id
    mod_pt.generation_config.decoder_start_token_id = _pt_id

except RuntimeError as e:
    if "CUDA out of memory" in str(e):
        print("⚠️ CUDA OOM moving EN→PT to GPU. Falling back to CPU.")
        mod_pt = mod_pt.to("cpu")
    else:
        raise

mod_pt.config.forced_bos_token_id = forced_bos_id(tok_pt, PT_LANG)
try: mod_pt.gradient_checkpointing_enable()
except Exception: pass

collator_pt = DataCollatorForSeq2Seq(tokenizer=tok_pt, model=mod_pt, padding="longest")
trainer_pt = Trainer(
    model=mod_pt,
    args=args_pt,
    train_dataset=train_pt_tok,
    eval_dataset=None,
    data_collator=collator_pt,
    tokenizer=tok_pt,
)

print("✅ EN→PT trainer ready. Starting training…")
train_output_pt = trainer_pt.train()
print("✅ EN→PT training done.")

print("🔎 Evaluating EN→PT…")
_ = evaluate_pair(mod_pt, tok_pt, valid_pt, name="EN→PT")

# Free VRAM
import gc
del trainer_pt, collator_pt, train_output_pt
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache(); torch.cuda.synchronize()


🔹 Loading EN→PT model...


C:\Users\jeeva\AppData\Local\Temp\ipykernel_36288\2925085539.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_pt = Trainer(


✅ EN→PT trainer ready. Starting training…


Step,Training Loss
100,1.133100
200,0.977600
300,0.852500
400,0.794800
500,0.760400


✅ EN→PT training done.
🔎 Evaluating EN→PT…
EN→PT -> BLEU 0.00 | chrF 0.86 | exact 0.00%  on 300 ex.


In [ ]:
@torch.inference_mode()
def translate_es(model, tok, text, max_new_tokens=128, num_beams=4):
    enc = tok([text], return_tensors="pt", padding=True, truncation=True, max_length=MAX_SRC_LEN).to(model.device)
    lang_id = forced_bos_id(tok, "spa_Latn")
    out = model.generate(
        **enc, num_beams=num_beams, max_new_tokens=max_new_tokens,
        no_repeat_ngram_size=3, early_stopping=True,
        forced_bos_token_id=lang_id,
        decoder_start_token_id=lang_id,   # << added
    )
    return tok.batch_decode(out, skip_special_tokens=True)[0]

@torch.inference_mode()
def translate_pt(model, tok, text, max_new_tokens=128, num_beams=4):
    enc = tok([text], return_tensors="pt", padding=True, truncation=True, max_length=MAX_SRC_LEN).to(model.device)
    lang_id = forced_bos_id(tok, "por_Latn")
    out = model.generate(
        **enc, num_beams=num_beams, max_new_tokens=max_new_tokens,
        no_repeat_ngram_size=3, early_stopping=True,
        forced_bos_token_id=lang_id,
        decoder_start_token_id=lang_id,   # << added
    )
    return tok.batch_decode(out, skip_special_tokens=True)[0]

print("ES:", translate_es(mod_es, tok_es, "This is a small test of a machine translation system."))
print("PT:", translate_pt(mod_pt, tok_pt, "How are you today?"))


ES: 
PT: 
